## The Three Editions

| Edition | What It Includes |
|---|---|
| **Core** | Streaming ingest only |
| **Pro** | Core + CDC (change data capture: you can update and delete rows in the target table) |
| **Advanced** | Pro + Expectations (a rule that says "this row must satisfy X to be considered valid") |

## The Three Actions

| Action | SQL | Python | Behavior |
|---|---|---|---|
| **WARN** | `ON VIOLATION WARN` | `@dlt.expect` | Keep the row, log a warning |
| **DROP ROW** | `ON VIOLATION DROP ROW` | `@dlt.expect_or_drop` | Silently remove the row |
| **FAIL UPDATE** | `ON VIOLATION FAIL UPDATE` | `@dlt.expect_or_fail` | Stop the entire pipeline |

In [0]:
CREATE OR REFRESH STREAMING TABLE clickstream_raw
COMMENT "Raw Wikipedia clickstream data."
AS
SELECT
  curr_title AS current_page_title,
  prev_title AS previous_page_title,
  CAST(n AS INT) AS click_count,
  type AS link_type
FROM STREAM read_files(
  "/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed-json",
  format => 'json'
);

CREATE OR REFRESH STREAMING TABLE clickstream_clean(
  -- Rule 1: Drop rows with missing page title
  CONSTRAINT valid_current_page 
    EXPECT (current_page_title IS NOT NULL) 
    ON VIOLATION DROP ROW,

  -- Rule 2: Fail the pipeline if click_count is invalid
  CONSTRAINT valid_click_count 
    EXPECT (click_count > 0) 
    ON VIOLATION FAIL UPDATE,

  -- Rule 3: Warn (but keep) rows with very short titles
  CONSTRAINT warn_short_page 
    EXPECT (LENGTH(current_page_title) > 3) 
    --ON VIOLATION WARN (warn is the default behavior when no action is specified.)
)
COMMENT "Cleaned clickstream with three expectation types."
AS
SELECT
  current_page_title,
  previous_page_title,
  click_count,
  link_type
FROM STREAM(live.clickstream_raw);

CREATE OR REFRESH MATERIALIZED VIEW top_clickstream_pages
COMMENT "Aggregated click counts."
AS
SELECT
  current_page_title,
  SUM(click_count) AS total_clicks,
  COUNT(*) AS link_types_count
FROM live.clickstream_clean
GROUP BY current_page_title;

In [0]:
-- Step 1: Create the target streaming table
CREATE OR REFRESH STREAMING TABLE customers_clean;

-- Step 2: Apply changes (THIS is CDC)
APPLY CHANGES INTO live.customers_clean
FROM STREAM(live.customers_raw)
KEYS (customer_id)
SEQUENCE BY updated_at
APPLY AS DELETE WHEN operation = 'DELETE'
COLUMNS * EXCEPT (operation, updated_at)
STORED AS SCD TYPE 1;